<p class="h1">ECE 447 - Notebook 10</p>
<p class="h2">Regularization</p>

In this section, we cover the following concept:

- regularization (L1 and L2).

# Regularization

When we train a model, the value of cost function (Log Loss) decreases; this is what we want. However, we are interested in the cost function value for the testing data. We want a model that is general enough and performs well on unseen data. If we train too little, our model is not able to capture the characteristic of the problem. We call this situation *underfitting*. If we train too much, the model is very good at predicting previously seen data but it is not as good on new data. In this case, we have *overfitting*.

To prevent overfitting, we could stop the minimization before the value of loss function is too low. The problem determining when to stop. 

Another approach is to penalize the complexity of the model. During the training process, we try to minimize the cost function $J(\theta)$. In this case, we modify the cost function to include penalization of the model's complexity:

$$
J(\theta) = Cost(\textrm{Features},\theta) + \lambda*penalty(\theta),
$$

where $\lambda$ controls the effect of the penalization on the cost function. We can observe that the penalization depends only of the model's parameters (hyperparameters). There are multiple forms of penalization function. Two of them are **Ridge** (L2 regularization) and **LASSO** (Least Absolute Shrinkage and Selection Operator) (L1 regularization).

**Ridge (L2 regularization)** tries to keep the values of hyperparameters small.
$$
\textrm{L2 regularization} : \lambda\sum_{j=1}^n\theta_j^2
$$

**LASSO (L1 regularization)** tries to keep small values of hyperparameters and to reduce their number.
$$
\textrm{L1 regularization} : \lambda\sum_{j=1}^n|\theta_j|
$$

In [ ]:
import numpy as np
import pandas as pd
import math
import pandas as pd

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, auc

from sklearn.datasets import make_classification
from sklearn.preprocessing import PolynomialFeatures

import matplotlib.pyplot as plt

In [ ]:
def sigmoid(z):
    return 1/(1+math.e**-z)

def decision(prob, boundary=0.5):
    if prob >= boundary:
        return 1
    return 0

def predict(Features, Weights):
    z = Features @ Weights
    return sigmoid(z)

def cost_function(Features, Labels, Weights):
    m = len(Labels)
    Ypred = predict(Features, Weights)
    # cost when label=1
    cost_class1 = -Labels * np.log(Ypred)
    # cost when label=0
    cost_class2 = -(1 - Labels) * np.log(1 - Ypred)
    tot_cost = cost_class1 + cost_class2
    tot_cost = tot_cost.sum()/m
    return tot_cost

def update_weights(Features, Labels, Weights, learning_rate):
    m = len(Features)
    Ypred = predict(Features, Weights)
    gradient = Features.T @ (Ypred - Labels)
    newWeights = Weights - learning_rate *(1/m) * gradient
    return newWeights

def classify(Ypreds, boundary=0.5):
    decision_bound = np.vectorize(decision)
    return decision_bound(Ypreds, boundary).flatten()

def train(Features, Labels, learning_rate, n_iterations):
    Weights = np.zeros((Features.shape[1],1))
    cost_history = []
    for i in range(n_iterations):
        Weights = update_weights(Features, Labels, Weights, learning_rate)
        cost = cost_function(Features, Labels, Weights)
        cost_history.append(cost)
    return Weights, cost_history

def classify_multi(Features,W):
    Yp = np.apply_along_axis(lambda w: predict(Features,w),0,W)
    return np.argmax(Yp,axis=1)

Let us take a look at a simple example. First we create a model without regularization.

In [ ]:
X, Y = make_classification(n_samples=100,  n_features=2, n_classes=2, random_state=3, n_redundant=0, n_informative=2)
# Polynomial degree to use
degree = 6


In [ ]:
plt.scatter(X[:,0][Y==0],X[:,1][Y==0])
plt.scatter(X[:,0][Y==1],X[:,1][Y==1])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split( X , Y , test_size = 0.4, random_state = 9)

plt.figure(figsize=(10,5))
plt.subplot(121)
plt.scatter(X_train[:,0],X_train[:,1],c=Y_train)
plt.title("train")

plt.subplot(122)
plt.scatter(X_test[:,0],X_test[:,1],c=Y_test)
plt.title("test");

In [ ]:
def plot_decision(X, Y, W, degree, title):
    xg = np.linspace(X[:,0].min(), X[:,0].max(), 150)
    yg = np.linspace(X[:,1].min(), X[:,1].max(), 150)
    xx, yy = np.meshgrid(xg,yg)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    poly_features = PolynomialFeatures(degree = degree)
    bias = np.ones(len(xx.ravel())).reshape(len(xx.ravel()),1)
    
    Xcont = poly_features.fit_transform(np.vstack((xx.ravel(),yy.ravel())).T)
    Xcont = np.hstack((bias,Xcont))
    
    Yp = np.apply_along_axis(lambda w: predict(Xcont,w),0,W)
    Yp = np.argmax(Yp,axis=1)
    Z = Yp.reshape(xx.shape)
    
    plt.contourf(xx,yy,Z,cmap=plt.cm.Paired)
    plt.scatter(X[:,0],X[:,1],c=Y)
    plt.title(title);

In [ ]:
def multiclassification(X, Y, n_cat, degree, lr=0.0001, iters=10000):
    poly_features = PolynomialFeatures(degree = degree)
    X_poly = poly_features.fit_transform(X)
    bias = np.ones(len(X))
    Features = np.vstack((bias,X_poly.T)).T
    weights = []
    for i in range(n_cat):
        Labels = np.zeros(len(Y))
        Labels[Y==i] = 1
        Labels = Labels.reshape(len(Y),1)
        w,h = train(Features, Labels, lr, iters)
        weights.append(w)
    W = np.hstack(weights)
    return classify_multi(Features, W),W

In [ ]:
%%time
Yp_train_noR, W_noR = multiclassification(X_train,Y_train,2,degree, lr=0.01, iters=20000)

In [ ]:
def eval_classification(X,W,degree):
    poly_features = PolynomialFeatures(degree = degree)
    X_poly = poly_features.fit_transform(X)
    bias = np.ones(len(X))
    Features = np.vstack((bias,X_poly.T)).T
    return classify_multi(Features, W)

In [ ]:
Yp_noR = eval_classification(X_test,W_noR,degree)

In [ ]:
print("Train accuracy:",accuracy_score(Y_train,Yp_train_noR))
print("Test accuracy:",accuracy_score(Y_test,Yp_noR))

In [ ]:
theta = W_noR[:,1]
# theta[np.abs(theta) < 1e-2] = 0
print("Theta values without regularization:\n",theta.reshape(len(theta),1))

## Ridge - L2 Regularization

We need to modify our cost function to include the regularization term.

In [ ]:
def cost_function_l2(Features, Labels, Weights, lda):
    m = len(Labels)
    Ypred = predict(Features, Weights)
    # cost when label=1
    cost_class1 = Labels*np.log(Ypred)
    # cost when label=0
    cost_class2 = (1 - Labels)*np.log(1 - Ypred)
    tot_cost = cost_class1 - cost_class2
    tot_cost = tot_cost.sum()/m
    # L2 regularization
    penalty = lda/(2) * (Weights**2).sum()
    return tot_cost + penalty

And we need to modify the gradient descent calculation.

In [ ]:
def update_weights_l2(Features, Labels, Weights, learning_rate, lda):
    m = len(Features)
    Ypred = predict(Features, Weights)
    gradient = Features.T @ (Ypred - Labels)
    gradient[1:] += (lda)* Weights[1:]
    newWeights = Weights - learning_rate *(1/m) * gradient
    return newWeights

In [ ]:
def train_l2(Features, Labels, learning_rate, n_iterations, lda):
    Weights = np.zeros((Features.shape[1],1))
    cost_history = []
    for i in range(n_iterations):
        Weights = update_weights_l2(Features, Labels, Weights, learning_rate, lda)
        cost = cost_function_l2(Features, Labels, Weights, lda)
        cost_history.append(cost)
    return Weights, cost_history

In [ ]:
def multiclassification_l2(X, Y, n_cat, degree, lr=0.1, iters=20000, lda=1):
    poly_features = PolynomialFeatures(degree = degree)
    X_poly = poly_features.fit_transform(X)
    bias = np.ones(len(X))
    Features = np.vstack((bias,X_poly.T)).T
    weights = []
    for i in range(n_cat):
        Labels = np.zeros(len(Y))
        Labels[Y==i] = 1
        Labels = Labels.reshape(len(Y),1)
        w,h = train_l2(Features, Labels, lr, iters, lda)
        weights.append(w)
    W = np.hstack(weights)
    return classify_multi(Features, W),W

In [ ]:
%time
Yp_train_L2,W_L2 = multiclassification_l2(X_train,Y_train,2,degree, lr=0.01, iters=20000, lda=0.9)

In [ ]:
Yp_L2 = eval_classification(X_test,W_L2,degree)

In [ ]:
print("Train accuracy:",accuracy_score(Y_train,Yp_train_L2))
print("Test accuracy:",accuracy_score(Y_test,Yp_L2))

In [ ]:
theta = W_L2[:,1]
# theta[np.abs(theta) < 1e-2] = 0
print("The regularized theta using ridge (L2):\n",theta.reshape(len(theta),1))

Let us compare the model hyperparameters values with values obtained for the model without regularization. As we can see, all the values are betwenn -1 and 1.

## LASSO - L1 Regularization

Now let us modify the cost function to include the L1 regularization term.

In [ ]:
def cost_function_l1(Features, Labels, Weights, lda):
    m = len(Labels)
    Ypred = predict(Features, Weights)
    # cost when label=1
    cost_class1 = Labels*np.log(Ypred)
    # cost when label=0
    cost_class2 = (1 - Labels)*np.log(1 - Ypred)
    tot_cost = cost_class1 - cost_class2
    tot_cost = tot_cost.sum()/m
    # L1 regularization
    penalty = lda * np.abs(Weights).sum()
    return tot_cost + penalty

In [ ]:
def update_weights_l1(Features, Labels, Weights, learning_rate, lda):
    m = len(Features)
    Ypred = predict(Features, Weights)
    gradient = Features.T @ (Ypred - Labels)
    # Adding L1 effect
    gradient[1:] += lda * (Weights[1:]/np.abs(Weights[1:]))
    newWeights = Weights - learning_rate *(1/m) * gradient
    return newWeights

In [ ]:
def train_l1(Features, Labels, learning_rate, n_iterations, lda):
    Weights = np.ones((Features.shape[1],1))
    cost_history = []
    for i in range(n_iterations):
        Weights = update_weights_l1(Features, Labels, Weights, learning_rate, lda)
        cost = cost_function_l1(Features, Labels, Weights, lda)
        cost_history.append(cost)
    return Weights, cost_history

In [ ]:
def multiclassification_l1(X, Y, n_cat, degree, lr=0.1, iters=20000, lda=1):
    poly_features = PolynomialFeatures(degree = degree)
    X_poly = poly_features.fit_transform(X)
    bias = np.ones(len(X))
    Features = np.vstack((bias,X_poly.T)).T
    weights = []
    for i in range(n_cat):
        Labels = np.zeros(len(Y))
        Labels[Y==i] = 1
        Labels = Labels.reshape(len(Y),1)
        w,h = train_l1(Features, Labels, lr, iters, lda)
        weights.append(w)
    W = np.hstack(weights)
    return classify_multi(Features, W),W

In [ ]:
%%time
Yp_train_L1,W_L1 = multiclassification_l1(X_train,Y_train,2,degree, lr=0.01, iters=20000, lda=0.9)

In [ ]:
Yp_L1 = eval_classification(X_test,W_L1,degree)

In [ ]:
print("Train accuracy:",accuracy_score(Y_train,Yp_train_L1))
print("Test accuracy:",accuracy_score(Y_test,Yp_L1))

In [ ]:
theta = W_L1[:,1]
theta[np.abs(theta) < 1e-5] = 0
print("The regularized theta using lasso regression:\n",theta.reshape(len(theta),1))

In this case, we see that some values are less than 0.00001; we consider them as zero.

Let us visually comparing the models' performances. We can observe the regions that each model is able to classify.

In [ ]:
plt.figure(figsize=(18,4))
plt.subplot(1,3,1)
plot_decision(X_train, Yp_train_noR, W_noR, degree,"No regularization (degree:{})".format(degree))
plt.subplot(1,3,2)
plot_decision(X_train, Yp_train_L2, W_L2, degree,"Using Ridge (L2) (degree:{})".format(degree))
plt.subplot(1,3,3)
plot_decision(X_train, Yp_train_L1, W_L1, degree, "Using Lasso (L1) (degree:{})".format(degree))


Let us compare the numeric results. We observe a reduction in the models' accuracy for the training data when regularization is used, but there is an increase in accuracy for the testing data. This means that the regularized models are mode general. 

In [ ]:
pd.DataFrame([[accuracy_score(Y_train,Yp_train_noR),accuracy_score(Y_test,Yp_noR)],
             [accuracy_score(Y_train,Yp_train_L2),accuracy_score(Y_test,Yp_L2)],
             [accuracy_score(Y_train,Yp_train_L1),accuracy_score(Y_test,Yp_L1)]],
             columns=["Train accuracity","Test accuracity"],index=["No regularization","Ridge (L2)","LASSO (L1)"])

## Regularization with scikit-learn

In `scikit-learn`, the logistic regression method uses L2 regularization by default. The type of regularization can be changed with the parameter `penalty`. Set `penalty="L2"` to use Ridge or `penalty="L1"` to use LASSO. 

In [ ]:
from sklearn.linear_model import LogisticRegression
poly_features = PolynomialFeatures(degree = degree)

Let us create a classification model using L2 regularization.

In [ ]:
logreg_L2 = LogisticRegression(penalty="l2")
X_poly = poly_features.fit_transform(X_train)
logreg_L2.fit(X_poly,Y_train)

Yp_train_L2 = logreg_L2.predict(X_poly)
X_poly = poly_features.fit_transform(X_test)
Yp_L2 = logreg_L2.predict(X_poly)
Yp_train_L2 = classify(Yp_train_L2)
Yp_L2 = classify(Yp_L2)
thetaRidge=logreg_L2.coef_
print("The regularized theta using Ridge regression:\n",thetaRidge.reshape(thetaRidge.shape[1],1))

And now, with L1 regularization.

In [ ]:
#logreg_L1 = LogisticRegression(penalty="l1")

logreg_L1 = LogisticRegression(C=1, penalty="l1", solver="liblinear")

X_poly = poly_features.fit_transform(X_train)
logreg_L1.fit(X_poly,Y_train)

Yp_train_L1 = logreg_L1.predict(X_poly)
X_poly = poly_features.fit_transform(X_test)
Yp_L1 = logreg_L1.predict(X_poly)
thetaLasso = logreg_L1.coef_
print("The regularized theta using Lasso regression:\n",thetaLasso.reshape(thetaLasso.shape[1],1))

In [ ]:
def plot_decision_sk(X, Yp, logreg, degree, title):
    xg = np.linspace(X[:,0].min(), X[:,0].max(), 150)
    yg = np.linspace(X[:,1].min(), X[:,1].max(), 150)

    xx, yy = np.meshgrid(xg,yg)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    poly_features = PolynomialFeatures(degree = degree)
    Xcont = poly_features.fit_transform(np.vstack((xx.ravel(),yy.ravel())).T)
    Z = logreg.predict(Xcont).reshape(xx.shape)
    plt.contourf(xx,yy,Z,levels=1)
    plt.scatter(X[:,0],X[:,1],c=Yp)
    plt.title(title);

Let us plot the decision boundaries for both models.

In [ ]:
plot_decision_sk(X_train,Yp_train_L2,logreg_L2,degree,'Ridge (L2)')
print("Train accuracy:",accuracy_score(Y_train,Yp_train_L2))
print("Test accuracy:",accuracy_score(Y_test,Yp_L2))

In [ ]:
plot_decision_sk(X_train,Yp_train_L1,logreg_L1,degree,'Lasso (L1)')
print("Train accuracy:",accuracy_score(Y_train,Yp_train_L1))
print("Test accuracy:",accuracy_score(Y_test,Yp_L1))

In [ ]:
pd.DataFrame([[accuracy_score(Y_train,Yp_train_noR),accuracy_score(Y_test,Yp_noR)],
             [accuracy_score(Y_train,Yp_train_L2),accuracy_score(Y_test,Yp_L2)],
             [accuracy_score(Y_train,Yp_train_L1),accuracy_score(Y_test,Yp_L1)]],
             columns=["Train accuracity","Test accuracity"],index=["No regularization","Ridge (L2)","LASSO (L1)"])

## Regularization and model coefficients

To control the regularization effect on the cost function, we use the parameter, $\lambda$. Any changes of $\lambda$ changes the behaviour of the model. The $\lambda=0$ removes the regularization effect, while high values of $\lambda$ make the model too general which can lead to low correctness of classification.

The following plots show the effect of $\lambda$ on the model's parameters. In `scikit-learn`, $\lambda$ is identified by the parameter `C`, but as the inverse of $\lambda$ (C=$1/\lambda$). This means that values of C close to zero imply a higher regularization effect. 

In [ ]:
ldas = np.linspace(0.001, 1, 100)
coefs = []
for a in ldas:
    ridge = LogisticRegression(penalty="l2", C=a, tol=1e-6, warm_start=True)
    X_poly = poly_features.fit_transform(X_train)
    ridge.fit(X_poly,Y_train)
    coefs.append(ridge.coef_.flatten())

In [ ]:
ax = plt.gca()
ax.plot(1/ldas, coefs)
ax.set_xscale('log')
plt.xlabel('$\lambda$')
plt.ylabel('theta')
plt.title('Ridge coefficients as a function of the regularization')
plt.show()

Now we can observe how the hyperparameters reduce their values when $\lambda$ becomes larger. 

In [ ]:
ldas = np.linspace(0.01, 10, 100)
coefs = []
for a in ldas:
    #ridge = LogisticRegression(penalty="l1", C=a, tol=1e-6, warm_start=True)
    ridge = LogisticRegression(C=a, penalty='l1',solver='liblinear',tol=1e-6, warm_start=True);
    X_poly = poly_features.fit_transform(X_train)
    ridge.fit(X_poly,Y_train)
    coefs.append(ridge.coef_.flatten())

In [ ]:
ax = plt.gca()
ax.plot(1/ldas, coefs)
ax.set_xscale('log')
plt.xlabel('$\lambda$')
plt.ylabel('theta')
plt.title('LASSO coefficients as a function of the regularization')
plt.show()

Here we observe decreasing values of the hyperparameters, as well as a reduction in the number of hyperparameters. 

<div class="alert alert-success">
<h2>Exercise</h2>

✏️ Evaluate different polynomial degrees to determine changes in the model performance when  L2 and L1 regularizations are considered. 
</div>

# Recap

In this section, we introduce logistic regression, a way to use linear regression to create a classification model. We present new metrics to measure the performance of a classification model: confusion matrix; accuracy; precision; recall and ROC curves. We show how to use a binary classification approach to implement a multiclass classification model using the One-vs-Rest technique. Finally, regularization is presented as a way to generalize a model and reduce the problem of overfitting.